# Compressor Model Training - RUL, Failure Probability, Failure Mode


In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBRegressor, XGBClassifier


In [2]:
telemetry = pd.read_csv('compressor_telemetry.csv')
labels = pd.read_csv('compressor_labels.csv')
df = telemetry.merge(labels, on=['timestamp','asset_id'])
df.head()


,asset_id,inlet_pressure,outlet_pressure,compression_ratio,air_flow_rate,bearing_temp,oil_temp,oil_pressure,vibration,motor_current,motor_voltage,power_consumption,timestamp,failure_mode,rul_days,failure_next_30_days
0,C-201,2.49,12.10,4.86,503.26,65.01,49.72,3.99,1.53,34.95,440.63,15.40,2026-06-10 10:39:14.144194,Bearing Failure,220.0,0
1,C-202,2.44,11.94,4.89,503.78,65.78,49.18,4.02,1.59,35.14,436.79,15.35,2026-06-10 10:39:14.144194,Oil Degradation,220.0,0
2,C-203,2.41,11.97,4.96,498.75,65.32,50.58,3.98,1.54,34.76,441.54,15.35,2026-06-10 10:39:14.144194,Air Leakage,220.0,0
3,C-204,2.54,11.93,4.70,499.55,64.57,49.57,4.00,1.43,34.84,439.20,15.30,2026-06-10 10:39:14.144194,Bearing Failure,220.0,0
4,C-205,2.52,12.07,4.79,497.36,65.06,48.20,3.98,1.46,35.05,440.85,15.45,2026-06-10 10:39:14.144194,Healthy,220.0,0


In [3]:
FEATURES = [
'inlet_pressure','outlet_pressure','compression_ratio',
'air_flow_rate','bearing_temp','oil_temp','oil_pressure',
'vibration','motor_current','motor_voltage','power_consumption'
]
X = df[FEATURES]


## Model 1 - RUL Prediction


In [4]:
y_rul = df['rul_days']
X_train,X_test,y_train,y_test = train_test_split(X,y_rul,test_size=0.2,random_state=42)
rul_model = XGBRegressor(n_estimators=300,max_depth=8,learning_rate=0.05,random_state=42)
rul_model.fit(X_train,y_train)
pred_rul = rul_model.predict(X_test)
mae = mean_absolute_error(y_test,pred_rul)
rmse = np.sqrt(mean_squared_error(y_test,pred_rul))
r2 = r2_score(y_test,pred_rul)
print('MAE:',mae)
print('RMSE:',rmse)
print('R2:',r2)
joblib.dump(rul_model,'compressor_rul_model.pkl')


MAE: 2.536633304250759
RMSE: 5.322073585353504
R2: 0.9936915892015138


['compressor_rul_model.pkl']

## Model 2 - Failure Probability


In [5]:
y_fail = df['failure_next_30_days']
X_train,X_test,y_train,y_test = train_test_split(X,y_fail,test_size=0.2,random_state=42)
failure_model = XGBClassifier(n_estimators=300,max_depth=8,learning_rate=0.05,random_state=42)
failure_model.fit(X_train,y_train)
pred_fail = failure_model.predict(X_test)
acc = accuracy_score(y_test,pred_fail)
prec = precision_score(y_test,pred_fail)
rec = recall_score(y_test,pred_fail)
f1 = f1_score(y_test,pred_fail)
print('Accuracy:',acc)
print('Precision:',prec)
print('Recall:',rec)
print('F1:',f1)
joblib.dump(failure_model,'compressor_failure_probability_model.pkl')


Accuracy: 0.9981327160493827
Precision: 0.9937092443421557
Recall: 0.9877983680317243
F1: 0.9907449900566009


['compressor_failure_probability_model.pkl']

## Model 3 - Failure Mode Classification


In [6]:
encoder = LabelEncoder()
y_mode = encoder.fit_transform(df['failure_mode'])
X_train,X_test,y_train,y_test = train_test_split(X,y_mode,test_size=0.2,random_state=42)
mode_model = XGBClassifier(n_estimators=300,max_depth=8,learning_rate=0.05,random_state=42)
mode_model.fit(X_train,y_train)
pred_mode = mode_model.predict(X_test)
mode_acc = accuracy_score(y_test,pred_mode)
print('Accuracy:',mode_acc)
joblib.dump(mode_model,'compressor_failure_mode_model.pkl')
joblib.dump(encoder,'compressor_failure_mode_encoder.pkl')


Accuracy: 0.9989737654320988


['compressor_failure_mode_encoder.pkl']

## Save Metrics CSV


In [7]:
rul_metrics = pd.DataFrame({'Metric':['MAE','RMSE','R2'],'Value':[mae,rmse,r2]})
rul_metrics.to_csv('compressor_rul_metrics.csv',index=False)
failure_metrics = pd.DataFrame({'Metric':['Accuracy','Precision','Recall','F1'],'Value':[acc,prec,rec,f1]})
failure_metrics.to_csv('compressor_failure_probability_metrics.csv',index=False)
mode_metrics = pd.DataFrame({'Metric':['Accuracy'],'Value':[mode_acc]})
mode_metrics.to_csv('compressor_failure_mode_metrics.csv',index=False)
feature_importance = pd.DataFrame({'Feature':FEATURES,'Importance':rul_model.feature_importances_})
feature_importance.to_csv('compressor_rul_feature_importance.csv',index=False)
print('All models and metrics saved successfully')


All models and metrics saved successfully
